# 02 · Feature engineering

Builds every **deterministic** feature (no labels involved) for train and test in one place and saves
`work/features.parquet` plus named column sets in `work/feature_sets.json`.
Features that use the default label (repeat-applicant history, neighbour default rate) must be computed
inside each cross-validation fold, so they live in notebook 04.

| Set | Content | Used by |
| --- | --- | --- |
| `raw` | the 32 supplied predictors, minimally cleaned | reference only |
| `base` | raw + affordability, timing, external-score and loan-term features (55 columns) | all models (the previous project's best set) |
| `recon` | 6 reconstructed external-score columns (prediction and residual per score) | CatBoost-recon model |
| `plus` | 20 population statistics computed over train **and** test | tested in notebook 04 |

In [1]:
import sys, json
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from src.utils import load_raw, WORK, SENTINEL_DAYS

train, test = load_raw()
both = pd.concat([train.assign(IS_TEST=0), test.assign(IS_TEST=1, TARGET=np.nan)], ignore_index=True)
print(train.shape, test.shape)

(171202, 34) (61500, 33)


## 1. Cleaning (the `raw` set)
* `DAYS_EMPLOYED = 365243` is a placeholder (mostly pensioners), not 1,000 years of employment → set to missing
  and keep a flag.
* Missing categories would become their own level `__MISSING__` (none of the supplied categoricals has gaps here,
  but the code stays safe for other data).
* Missing numbers stay `NaN`: the tree models handle them natively and the neural network gets explicit flags.
* `SK_ID_CURR` is only an identifier and is never a feature.

In [2]:
RAW = [c for c in train.columns if c not in ('SK_ID_CURR', 'TARGET')]
X = both[RAW].copy()
X['DAYS_EMPLOYED'] = X.DAYS_EMPLOYED.replace(SENTINEL_DAYS, np.nan).astype(float)
CATEGORICAL = list(X.select_dtypes(exclude='number').columns)
for c in CATEGORICAL:
    X[c] = X[c].fillna('__MISSING__').astype(str)
print(len(RAW), 'raw predictors;', len(CATEGORICAL), 'categorical:', CATEGORICAL)

def ratio(a, b):
    return a / b.replace(0, np.nan)

32 raw predictors; 8 categorical: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


## 2. The `base` set
These are the feature groups that survived validation in the earlier rounds of this project.

**Affordability.** Income/credit/annuity ratios, income per family member and the goods-price gap
(credit above the goods price is financed fees/insurance).

**Timing.** Age at employment start and employment length as a share of age.

**External scores.** `EXT_SOURCE_1..3` are by far the strongest predictors. Their mean/min/max/std
and the number available summarise them; they are often missing, so summaries use whatever is present.

**Loan term and implied interest rate.** `AMT_CREDIT / AMT_ANNUITY` is the number of payments the credit
would take without interest; with interest it is lower than the real term, by an amount that depends on the rate.
Its distribution is spiky (standard products, notebook 01), so the signed and absolute distance to the
nearest multiple of 6 or 12 and the rounded value help separate products and price levels. Solving the annuity
formula for the rate under an assumed term (24/36/60 payments) gives a monotone "price of credit" proxy;
it is *not* the true contract rate.

In [3]:
B = X.copy()
B['EMPLOYED_SENTINEL'] = both.DAYS_EMPLOYED.eq(SENTINEL_DAYS).astype('int8')
B['MISSING_COUNT'] = both[RAW].isna().sum(axis=1)
B['CREDIT_TERM'] = ratio(B.AMT_CREDIT, B.AMT_ANNUITY)
B['CREDIT_GOODS_DIFF_RATIO'] = ratio(B.AMT_CREDIT - B.AMT_GOODS_PRICE, B.AMT_GOODS_PRICE)
B['INCOME_PER_PERSON'] = ratio(B.AMT_INCOME_TOTAL, B.CNT_FAM_MEMBERS)
B['INCOME_OVER_ANNUITY'] = ratio(B.AMT_INCOME_TOTAL, B.AMT_ANNUITY)
B['INCOME_OVER_CREDIT'] = ratio(B.AMT_INCOME_TOTAL, B.AMT_CREDIT)
B['EMPLOYED_OVER_AGE'] = ratio(B.DAYS_EMPLOYED, B.DAYS_BIRTH)
B['AGE_AT_EMPLOYMENT'] = (-B.DAYS_BIRTH + B.DAYS_EMPLOYED) / 365.25
ext = B[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']]
for agg in ['mean', 'min', 'max', 'std']:
    B['EXT_' + agg.upper()] = getattr(ext, agg)(axis=1)
B['EXT_COUNT'] = ext.notna().sum(axis=1)


def assumed_rate(payment_share, n):
    """Per-period rate r solving annuity/credit = r / (1 - (1+r)^-n), by bisection; NaN when impossible."""
    p = payment_share.to_numpy()
    low, high = np.zeros(len(p)), np.ones(len(p))
    for _ in range(40):
        mid = (low + high) / 2
        too_low = mid / (-np.expm1(-n * np.log1p(mid))) < p
        low, high = np.where(too_low, mid, low), np.where(too_low, high, mid)
    return np.where(np.isfinite(p) & (p >= 1 / n) & (p < 1), (low + high) / 2, np.nan)


share = ratio(B.AMT_ANNUITY, B.AMT_CREDIT)
for n in [24, 36, 60]:
    B[f'ASSUMED_RATE_{n}'] = assumed_rate(share, n)
for n in [6, 12]:
    z = B.CREDIT_TERM / n
    B[f'TERM_MULTIPLE_{n}_DISTANCE'] = (z - np.rint(z)).abs()
    B[f'TERM_MULTIPLE_{n}_SIGNED'] = z - np.rint(z)
    B[f'TERM_ROUND_{n}'] = np.rint(z).astype('Int64').astype(str)   # categorical: rounded number of 6/12-month blocks
B = B.replace([np.inf, -np.inf], np.nan)
BASE = list(B.columns)
print(len(BASE), 'base columns')

55 base columns


## 3. Reconstructed external scores (`recon`)
`EXT_SOURCE_1` is missing for 69% of applicants and `EXT_SOURCE_3` for 32%. A LightGBM regressor learns
each score from the other supplied columns. It gives (a) an estimate where the score is missing and
(b) where it is present, the **residual** "observed − expected", i.e. whether the bureau rates the applicant
better or worse than their profile suggests. Each score's regressor excludes that score itself.

This uses no default labels, so it is fitted on train + test together. Predictions for observed rows are
out-of-fold (5 folds) so that a row's residual never comes from a model that saw its own score.

In [4]:
import lightgbm as lgb
from sklearn.model_selection import KFold

raw_codes = X.copy()
for c in CATEGORICAL:
    raw_codes[c] = raw_codes[c].astype('category')
quality = {}
R = pd.DataFrame(index=X.index)
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    inputs, target = raw_codes.drop(columns=col), X[col].to_numpy()
    observed = np.flatnonzero(np.isfinite(target))
    pred, unobserved = np.zeros(len(X)), np.flatnonzero(~np.isfinite(target))
    for k, (a, b) in enumerate(KFold(5, shuffle=True, random_state=0).split(observed)):
        model = lgb.LGBMRegressor(n_estimators=300, learning_rate=.04, num_leaves=15, min_child_samples=100,
                                  reg_lambda=20, random_state=k, verbosity=-1, n_jobs=8)
        model.fit(inputs.iloc[observed[a]], target[observed[a]])
        pred[observed[b]] = model.predict(inputs.iloc[observed[b]])
        pred[unobserved] += model.predict(inputs.iloc[unobserved]) / 5
    R['RECON_' + col] = pred
    R['RESID_' + col] = target - pred                       # NaN where the score is genuinely missing
    resid = target[observed] - pred[observed]
    quality[col] = round(float(1 - resid.var() / target[observed].var()), 4)
RECON = list(R.columns)
print('out-of-fold R^2 of each reconstruction:', quality)

out-of-fold R^2 of each reconstruction: {'EXT_SOURCE_1': 0.4901, 'EXT_SOURCE_2': 0.2412, 'EXT_SOURCE_3': 0.1226}


## 4. New: population statistics over train + test (`plus`)
The rules forbid outside data, but `test.csv` is supplied data. Statistics that never touch the
label may use all 232,702 applicants. They describe how typical an applicant is for their group:

* **Frequencies** of organisation/occupation types.
* **Group-relative amounts**: income, credit, term and external-score mean divided by the median of the
  applicant's organisation, occupation and income-type group.
* **Value popularity**: how many applicants share the exact credit amount, annuity, goods price or
  (credit, annuity) pair. Popular combinations correspond to standard products.
* **Repeat applicants**: the `DAYS_*` columns are counted back from the application date, so the same
  person applying twice gets shifted values, but *differences* such as birth − ID-publication stay constant.
  `APPLICANT_KEY` = (gender, birth − ID publish, birth − registration) groups probable repeat applications
  (notebook 01). Here we only count group size and order; their labels are used in notebook 04.

In [5]:
P = B.copy()
for c in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']:
    P[c + '_FREQ'] = P[c].map(P[c].value_counts(normalize=True))
for group in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE']:
    for value in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CREDIT_TERM', 'EXT_MEAN']:
        P[f'{value}_REL_{group}'] = ratio(P[value], P.groupby(group)[value].transform('median'))
for cols in [['AMT_CREDIT'], ['AMT_ANNUITY'], ['AMT_GOODS_PRICE'], ['AMT_CREDIT', 'AMT_ANNUITY']]:
    P['COUNT_' + '_'.join(c.replace('AMT_', '') for c in cols)] = P.groupby(cols, dropna=False)[cols[0]].transform('size')

key = (both.CODE_GENDER.astype(str) + '|' + (both.DAYS_BIRTH - both.DAYS_ID_PUBLISH).astype(str) + '|'
       + (both.DAYS_BIRTH - both.DAYS_REGISTRATION).round().astype('Int64').astype(str))
P['REPEAT_COUNT'] = key.map(key.value_counts())
# Later applications of the same person have older DAYS_BIRTH (more negative); order 0 = first application.
P['REPEAT_ORDER'] = both.DAYS_BIRTH.groupby(key).rank(method='first', ascending=False) - 1
P = P.replace([np.inf, -np.inf], np.nan)
PLUS = [c for c in P.columns if c not in BASE]
print(len(PLUS), 'population columns;', f'{(P.REPEAT_COUNT > 1).mean():.1%} of applicants share an APPLICANT_KEY')

20 population columns; 4.1% of applicants share an APPLICANT_KEY


## 5. Save

In [6]:
out = pd.concat([P, R], axis=1).assign(SK_ID_CURR=both.SK_ID_CURR, TARGET=both.TARGET, IS_TEST=both.IS_TEST, APPLICANT_KEY=key)
out.to_parquet(WORK / 'features.parquet', index=False)
sets = dict(raw=RAW, base=BASE, recon=RECON, plus=PLUS, categorical=CATEGORICAL + ['TERM_ROUND_6', 'TERM_ROUND_12'])
(WORK / 'feature_sets.json').write_text(json.dumps(sets, indent=1))
pd.Series({k: len(v) for k, v in sets.items()}, name='columns')

raw            32
base           55
recon           6
plus           20
categorical    10
Name: columns, dtype: int64

Sanity checks: no label or ID column is a feature, and train/test have identical schemas.

In [7]:
assert not {'TARGET', 'SK_ID_CURR', 'IS_TEST', 'APPLICANT_KEY'} & set(BASE + RECON + PLUS)
assert len(out) == len(train) + len(test) and out.SK_ID_CURR.is_unique
assert (out[out.IS_TEST == 1].SK_ID_CURR.to_numpy() == test.SK_ID_CURR.to_numpy()).all()
out[RECON + PLUS].describe().T[['mean', 'min', 'max']].round(3)

,mean,min,max
RECON_EXT_SOURCE_1,0.528,0.146,0.864
RESID_EXT_SOURCE_1,0.000,-0.697,0.651
RECON_EXT_SOURCE_2,0.512,0.132,0.743
RESID_EXT_SOURCE_2,-0.000,-0.678,0.523
RECON_EXT_SOURCE_3,0.507,0.213,0.691
RESID_EXT_SOURCE_3,-0.000,-0.674,0.523
ORGANIZATION_TYPE_FREQ,0.108,0.000,0.222
OCCUPATION_TYPE_FREQ,0.162,0.002,0.311
AMT_INCOME_TOTAL_REL_ORGANIZATION_TYPE,1.160,0.162,742.857
AMT_CREDIT_REL_ORGANIZATION_TYPE,1.177,0.079,9.000
